# 01 - Bronze Extract

Ingesta RAW → Bronze para el caso fintech.

Datasets:
- `customers`
- `transactions`
- `fraud_alerts`

La lectura se realiza desde ADLS Gen2 mediante `abfss://...`, compatible con Managed Identity configurada en Databricks/Unity Catalog.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

dbutils.widgets.text("catalog_name", "fintech_lakehouse")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("raw_base_path", "abfss://raw@<storage-account>.dfs.core.windows.net/fintech")

catalog_name = dbutils.widgets.get("catalog_name")
bronze_schema = dbutils.widgets.get("bronze_schema")
raw_base_path = dbutils.widgets.get("raw_base_path").rstrip("/")

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {bronze_schema}")

In [ ]:
def read_raw_csv(dataset_name: str):
    path = f"{raw_base_path}/{dataset_name}/{dataset_name}.csv"
    return (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("multiLine", "true")
        .option("escape", '"')
        .csv(path)
        .withColumn("_source_file", F.input_file_name())
        .withColumn("_ingestion_ts", F.current_timestamp())
    )

customers_bronze = read_raw_csv("customers")
transactions_bronze = read_raw_csv("transactions")
fraud_alerts_bronze = read_raw_csv("fraud_alerts")

print("Customers:", customers_bronze.count())
print("Transactions:", transactions_bronze.count())
print("Fraud alerts:", fraud_alerts_bronze.count())

In [ ]:
customers_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("customers_bronze")
transactions_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("transactions_bronze")
fraud_alerts_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("fraud_alerts_bronze")

display(spark.sql("SHOW TABLES"))